# ERC-20: data sources

Section 4.2 and Table 8. Four combinations of a scam list and a legitimate list are compared with the selected features. In each scenario a Balanced Random Forest is tuned by stratified five-fold cross-validation on macro-F1 over the training partition (80%) and scored on the test partition (20%). Scenarios are ranked by $(\text{AUC-PR} + F_2)/2$, with scam as the positive class.

For reference the same procedure is repeated on all non-constant descriptors.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import json

import joblib
import numpy as np
import pandas as pd
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

from bfx.modeling import SEED, harmonise, scores

DATA = ROOT / "data" / "erc20"
RESULTS = ROOT / "results" / "erc20"
MODELS = ROOT / "models" / "erc20"
MODELS.mkdir(parents=True, exist_ok=True)
SELECTED = pd.read_csv(RESULTS / "selected_features.csv")["feature"].tolist()

SOURCES = {
    "CA": ("chainabuse_scam_tokens", 1),
    "TO": ("tokenscout_scam_tokens", 1),
    "CMC": ("cmc_licit_tokens", 0),
    "TS": ("tokenscout_licit_tokens", 0),
}


def load(name):
    file, _ = SOURCES[name]
    df = pd.read_csv(DATA / "features" / f"{file}.csv")
    address = df["token_file"].str.removesuffix(".csv").str.rsplit("_", n=1).str[-1].str.lower()
    return pd.concat([df, address.rename("address")], axis=1)


def scenario(fraud, licit):
    data = harmonise({fraud: load(fraud), licit: load(licit)}, fraud_sources=[fraud], key="address")
    X = data.drop(columns=["token_file", "label", "source_directory", "address", "source", "y"], errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    limit = np.finfo(np.float32).max  # tree models work in float32
    return X.clip(-limit, limit), data["y"]

In [ ]:
GRID = {"n_estimators": [200, 300], "max_depth": [20, None], "min_samples_leaf": [1, 2]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def run(fraud, licit, columns=None):
    X, y = scenario(fraud, licit)
    X = X[columns] if columns else X.loc[:, X.nunique() > 1]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    search = GridSearchCV(BalancedRandomForestClassifier(random_state=SEED, n_jobs=-1), GRID,
                          scoring="f1_macro", cv=cv, n_jobs=-1).fit(X_train, y_train)
    model = search.best_estimator_
    s = scores(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1])
    return model, {"scenario": f"{fraud} + {licit}", "tokens": len(X), "features": X.shape[1],
                   "params": search.best_params_, **s}


rows = []
for fraud, licit in [("CA", "CMC"), ("TO", "CMC"), ("CA", "TS"), ("TO", "TS")]:
    model, row = run(fraud, licit, SELECTED)
    joblib.dump(model, MODELS / f"{fraud}_{licit}.joblib")
    rows.append(row)
table8 = pd.DataFrame(rows)
table8["rank_score"] = (table8["auc_pr"] + table8["f2_fraud"]) / 2
table8 = table8.sort_values("rank_score", ascending=False).reset_index(drop=True)
table8.to_csv(RESULTS / "table8_scenarios.csv", index=False)

## Table 8 (selected features)

In [ ]:
table8.set_index("scenario")[["tokens", "f1_fraud", "balanced_accuracy", "roc_auc", "fpr", "f1", "rank_score"]].round(3)

Hyperparameters chosen by cross-validation:

In [ ]:
table8.set_index("scenario")["params"]

## Reference: all descriptors

In [ ]:
full = pd.DataFrame([run(f, l)[1] for f, l in [("CA", "CMC"), ("TO", "CMC"), ("CA", "TS"), ("TO", "TS")]])
full["rank_score"] = (full["auc_pr"] + full["f2_fraud"]) / 2
full.to_csv(RESULTS / "table8_all_descriptors.csv", index=False)
full.set_index("scenario")[["features", "f1_fraud", "balanced_accuracy", "roc_auc", "fpr", "rank_score"]].round(3)